# Expanded-label model: tracking QC and preliminary behavior

This notebook analyzes prediction H5 files from `mobile_worklabels_20260927_210240`.
It makes two layers:

- **Tracking QC:** visibility, likelihood, body-center coverage, jump flags.
- **Preliminary behavior:** locomotor activity, rest/active time, movement bouts,
  and pixel-distance summaries.

Behavior metrics are intentionally pixel-space and non-zone-based. They are suitable
for a preliminary committee update, but not yet for calibrated distance or
arena-zone interpretation.



In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import math
import pickle
import re

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass



In [ ]:
ROOT = Path('/content/drive/MyDrive/sp1DLC')
RUN = ROOT / 'Colab_Training_Runs/mobile_worklabels_20260927_210240'
PREDICTIONS = RUN / 'videos'

OUT = RUN / 'Tracking_Behavior_Analysis' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=True)

FOCUS = ['nose', 'left_ear', 'right_ear', 'spine_mid', 'left_hip', 'right_hip', 'tail_base']
TORSO = ['spine_mid', 'left_hip', 'right_hip', 'tail_base']

LIKELIHOOD_CUTOFF = 0.20
MAX_INTERPOLATION_GAP_FRAMES = 6
SMOOTHING_WINDOW_FRAMES = 5

# Behavior thresholds are data-driven and conservative. They avoid absolute unit claims.
REST_SPEED_QUANTILE = 0.20
ACTIVE_SPEED_QUANTILE = 0.50
MIN_BOUT_FRAMES = 6

plt.rcParams.update({
    'figure.dpi': 180,
    'savefig.dpi': 180,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.18,
    'axes.facecolor': '#fbfbf8',
    'figure.facecolor': 'white',
})

print('Prediction folder:', PREDICTIONS)
print('Output folder:', OUT)



In [ ]:
def natural_key(path: Path) -> tuple:
    return tuple(int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', path.name))


def h5_files() -> list[Path]:
    files = sorted(PREDICTIONS.glob('*.h5'), key=natural_key)
    files = [path for path in files if 'snapshot' in path.name or 'DLC_' in path.name]
    assert files, f'No prediction H5 files found in {PREDICTIONS}. Run Colab_Worklabels_Videos.ipynb first.'
    return files


def read_prediction(path: Path) -> pd.DataFrame:
    df = pd.read_hdf(path)
    assert isinstance(df.columns, pd.MultiIndex), f'{path.name} does not have DLC MultiIndex columns'
    missing = [part for part in FOCUS if part not in df.columns.get_level_values('bodyparts')]
    assert not missing, f'{path.name} is missing bodyparts: {missing}'
    return df.loc[:, df.columns.get_level_values('bodyparts').isin(FOCUS)]


def read_fps(path: Path) -> float | None:
    meta_path = path.with_name(path.stem + '_meta.pickle')
    if not meta_path.exists():
        return None
    try:
        with meta_path.open('rb') as handle:
            meta = pickle.load(handle)
    except Exception:
        return None
    stack = [meta]
    while stack:
        item = stack.pop()
        if isinstance(item, dict):
            for key, value in item.items():
                if str(key).lower() in {'fps', 'framerate', 'frame_rate'}:
                    try:
                        fps = float(value)
                        if math.isfinite(fps) and fps > 0:
                            return fps
                    except Exception:
                        pass
                elif isinstance(value, (dict, list, tuple)):
                    stack.append(value)
        elif isinstance(item, (list, tuple)):
            stack.extend(item)
    return None


def video_name(path: Path) -> str:
    return path.name.partition('DLC_')[0]


def get_coord(df: pd.DataFrame, bodypart: str, coord: str) -> pd.Series:
    cols = [
        col for col in df.columns
        if col[df.columns.names.index('bodyparts')] == bodypart
        and col[df.columns.names.index('coords')] == coord
    ]
    assert len(cols) == 1, f'Expected exactly one {bodypart}/{coord} column'
    return pd.to_numeric(df[cols[0]], errors='coerce')


def point_table(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    for part in FOCUS:
        likelihood = get_coord(df, part, 'likelihood')
        valid = likelihood >= LIKELIHOOD_CUTOFF
        out[part] = pd.DataFrame({
            'x': get_coord(df, part, 'x').where(valid),
            'y': get_coord(df, part, 'y').where(valid),
            'likelihood': likelihood,
            'valid': valid,
        })
    return out


def clean_series(series: pd.Series) -> pd.Series:
    filled = series.interpolate(limit=MAX_INTERPOLATION_GAP_FRAMES, limit_direction='both')
    if SMOOTHING_WINDOW_FRAMES > 1:
        filled = filled.rolling(SMOOTHING_WINDOW_FRAMES, center=True, min_periods=1).median()
    return filled


def body_center(points: dict[str, pd.DataFrame]) -> pd.DataFrame:
    x = pd.concat([points[part]['x'].rename(part) for part in TORSO], axis=1).mean(axis=1, skipna=True)
    y = pd.concat([points[part]['y'].rename(part) for part in TORSO], axis=1).mean(axis=1, skipna=True)
    valid_count = pd.concat([points[part]['valid'].rename(part) for part in TORSO], axis=1).sum(axis=1)
    center = pd.DataFrame({'x_raw': x, 'y_raw': y, 'valid_torso_points': valid_count})
    center['x'] = clean_series(center['x_raw'])
    center['y'] = clean_series(center['y_raw'])
    center['usable'] = center[['x', 'y']].notna().all(axis=1)
    center['step_px'] = np.hypot(center['x'].diff(), center['y'].diff()).where(center['usable'])
    center.loc[~center['usable'], 'step_px'] = np.nan
    return center


def count_bouts(mask: pd.Series, min_frames: int = MIN_BOUT_FRAMES) -> int:
    values = mask.fillna(False).astype(bool).to_numpy()
    count = 0
    run = 0
    for value in values:
        if value:
            run += 1
        else:
            if run >= min_frames:
                count += 1
            run = 0
    if run >= min_frames:
        count += 1
    return count



In [ ]:
raw = {}
for path in h5_files():
    name = video_name(path)
    df = read_prediction(path)
    points = point_table(df)
    center = body_center(points)
    fps = read_fps(path)
    raw[name] = {'path': path, 'df': df, 'points': points, 'center': center, 'fps': fps}

all_steps = pd.concat([item['center']['step_px'] for item in raw.values()], ignore_index=True).dropna()
rest_threshold = float(all_steps.quantile(REST_SPEED_QUANTILE))
active_threshold = float(all_steps.quantile(ACTIVE_SPEED_QUANTILE))
print('Videos:', len(raw))
print('Rest threshold px/frame:', round(rest_threshold, 3))
print('Active threshold px/frame:', round(active_threshold, 3))



In [ ]:
tracking_rows = []
behavior_rows = []
qc_rows = []

for name, item in raw.items():
    center = item['center']
    fps = item['fps']
    usable = center['usable']
    steps = center['step_px']
    resting = steps <= rest_threshold
    active = steps >= active_threshold
    frame_count = len(center)

    tracking_rows.append({
        'video': name,
        'frames': frame_count,
        'fps_from_metadata': fps,
        'usable_center_pct': float(usable.mean() * 100),
        'median_step_px_per_frame': float(steps.median(skipna=True)),
        'p95_step_px_per_frame': float(steps.quantile(0.95)),
        'max_step_px_per_frame': float(steps.max(skipna=True)),
        'total_distance_px': float(steps.sum(skipna=True)),
    })

    behavior = {
        'video': name,
        'frames': frame_count,
        'usable_center_pct': float(usable.mean() * 100),
        'total_distance_px': float(steps.sum(skipna=True)),
        'rest_frames_pct': float(resting.mean() * 100),
        'active_frames_pct': float(active.mean() * 100),
        'rest_bouts_count': count_bouts(resting),
        'active_bouts_count': count_bouts(active),
        'median_step_px_per_frame': float(steps.median(skipna=True)),
        'p95_step_px_per_frame': float(steps.quantile(0.95)),
    }
    if fps:
        behavior['duration_s'] = frame_count / fps
        behavior['total_distance_px_per_min'] = behavior['total_distance_px'] / max(frame_count / fps / 60, 1e-9)
        behavior['median_speed_px_s'] = behavior['median_step_px_per_frame'] * fps
        behavior['p95_speed_px_s'] = behavior['p95_step_px_per_frame'] * fps
    behavior_rows.append(behavior)

    for part in FOCUS:
        point = item['points'][part]
        qc_rows.append({
            'video': name,
            'bodypart': part,
            'visible_pct': float(point['valid'].mean() * 100),
            'median_likelihood': float(point['likelihood'].median(skipna=True)),
        })

tracking_df = pd.DataFrame(tracking_rows).sort_values('video', key=lambda s: s.map(lambda value: natural_key(Path(str(value)))))
behavior_df = pd.DataFrame(behavior_rows).sort_values('video', key=lambda s: s.map(lambda value: natural_key(Path(str(value)))))
qc_df = pd.DataFrame(qc_rows)

tracking_df.to_csv(OUT / 'tracking_qc_summary.csv', index=False)
behavior_df.to_csv(OUT / 'preliminary_behavior_summary.csv', index=False)
qc_df.to_csv(OUT / 'bodypart_visibility_qc.csv', index=False)

display(tracking_df.round(2))
display(behavior_df.round(2))



In [ ]:
def save_bar(values: pd.Series, title: str, xlabel: str, filename: str, color: str) -> None:
    fig_h = max(5.5, 0.22 * len(values) + 2.0)
    fig, ax = plt.subplots(figsize=(9.5, fig_h))
    ordered = values.sort_values()
    ax.barh(ordered.index, ordered.values, color=color, alpha=0.9)
    ax.set_title(title, loc='left', fontsize=14, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    fig.tight_layout()
    fig.savefig(OUT / filename, bbox_inches='tight')
    plt.show()


save_bar(tracking_df.set_index('video')['usable_center_pct'], 'Tracking coverage', '% usable body-center frames', 'tracking_coverage.png', '#2f6f73')
save_bar(behavior_df.set_index('video')['total_distance_px'], 'Preliminary locomotion', 'Total body-center distance, pixels', 'behavior_total_distance.png', '#6f5d2f')
save_bar(behavior_df.set_index('video')['active_frames_pct'], 'Activity level', '% frames above data-driven active threshold', 'behavior_active_frames.png', '#654b7c')
save_bar(behavior_df.set_index('video')['rest_frames_pct'], 'Rest/low-movement time', '% frames below data-driven rest threshold', 'behavior_rest_frames.png', '#84693d')



In [ ]:
qc_heat = qc_df.pivot(index='video', columns='bodypart', values='visible_pct')
qc_heat = qc_heat.loc[tracking_df['video'], FOCUS]

fig, ax = plt.subplots(figsize=(10.5, max(6, 0.24 * len(qc_heat) + 1.6)))
image = ax.imshow(qc_heat.values, aspect='auto', vmin=0, vmax=100, cmap='viridis')
ax.set_title('Bodypart visibility', loc='left', fontsize=14, weight='bold')
ax.set_xticks(np.arange(len(FOCUS)))
ax.set_xticklabels(FOCUS, rotation=35, ha='right')
ax.set_yticks(np.arange(len(qc_heat.index)))
ax.set_yticklabels(qc_heat.index)
for row in range(qc_heat.shape[0]):
    for col in range(qc_heat.shape[1]):
        value = qc_heat.iat[row, col]
        ax.text(col, row, f'{value:.0f}', ha='center', va='center',
                color='white' if value < 65 else '#14201e', fontsize=7)
cbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('% frames at/above cutoff')
fig.tight_layout()
fig.savefig(OUT / 'tracking_bodypart_visibility_heatmap.png', bbox_inches='tight')
plt.show()



In [ ]:
def plot_video_page(name: str, item: dict, pdf: PdfPages | None = None) -> None:
    center = item['center']
    ok = center['usable']
    fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
    axes = axes.ravel()

    ax = axes[0]
    if ok.any():
        ax.plot(center.loc[ok, 'x'], center.loc[ok, 'y'], color='#2f6f73', lw=1.0)
        ax.scatter(center.loc[ok, 'x'].iloc[:1], center.loc[ok, 'y'].iloc[:1], s=36, color='#1b4332', label='start')
        ax.scatter(center.loc[ok, 'x'].iloc[-1:], center.loc[ok, 'y'].iloc[-1:], s=36, color='#b84a3a', label='end')
    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'{name}: body-center path', loc='left', weight='bold')
    ax.set_xlabel('x pixel')
    ax.set_ylabel('y pixel')
    ax.legend(frameon=False, fontsize=8)

    ax = axes[1]
    ax.plot(center.index, center['step_px'], color='#654b7c', lw=0.8)
    ax.axhline(rest_threshold, color='#84693d', ls='--', lw=1, label='rest threshold')
    ax.axhline(active_threshold, color='#b84a3a', ls='--', lw=1, label='active threshold')
    ax.set_title('Movement over time', loc='left', weight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Pixels/frame')
    ax.legend(frameon=False, fontsize=8)

    ax = axes[2]
    rolling = center['step_px'].rolling(60, min_periods=1).mean()
    ax.plot(center.index, rolling, color='#6f5d2f', lw=1.0)
    ax.set_title('Rolling activity', loc='left', weight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Rolling mean px/frame')

    ax = axes[3]
    labels = ['usable', 'rest', 'active']
    values = [ok.mean() * 100, (center['step_px'] <= rest_threshold).mean() * 100, (center['step_px'] >= active_threshold).mean() * 100]
    ax.bar(labels, values, color=['#2f6f73', '#84693d', '#654b7c'])
    ax.set_ylim(0, 100)
    ax.set_title('Frame-level summaries', loc='left', weight='bold')
    ax.set_ylabel('% frames')

    fig.tight_layout()
    if pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.show()


pdf_path = OUT / 'tracking_behavior_video_pages.pdf'
with PdfPages(pdf_path) as pdf:
    for name in tracking_df['video']:
        plot_video_page(name, raw[name], pdf=pdf)
print('Saved:', pdf_path)



In [ ]:
flags = tracking_df[['video', 'usable_center_pct', 'p95_step_px_per_frame', 'max_step_px_per_frame']].copy()
flags['review_reason'] = ''
flags.loc[flags['usable_center_pct'] < 90, 'review_reason'] += 'tracking coverage below 90%; '
flags.loc[flags['p95_step_px_per_frame'] > tracking_df['p95_step_px_per_frame'].median() * 2, 'review_reason'] += 'large high-percentile movement jumps; '
flags = flags[flags['review_reason'].str.len() > 0]
flags.to_csv(OUT / 'manual_review_flags.csv', index=False)
display(flags.round(2))

report = OUT / 'tracking_behavior_report.md'
with report.open('w', encoding='utf-8') as handle:
    handle.write('# Expanded-label tracking and preliminary behavior report\n\n')
    handle.write(f'Generated: {datetime.now(timezone.utc).isoformat()}\n\n')
    handle.write(f'Run folder: `{RUN}`\n\n')
    handle.write('## Scope\n\n')
    handle.write('This report uses DeepLabCut predictions from the expanded-label model. Tracking QC and behavior summaries are preliminary. Behavior metrics are pixel-space locomotor/activity measures, not calibrated arena-zone measures.\n\n')
    handle.write('## Settings\n\n')
    handle.write(f'- Likelihood cutoff: {LIKELIHOOD_CUTOFF}\n')
    handle.write(f'- Short-gap interpolation: up to {MAX_INTERPOLATION_GAP_FRAMES} frames\n')
    handle.write(f'- Smoothing window: {SMOOTHING_WINDOW_FRAMES} frames\n')
    handle.write(f'- Rest threshold: {rest_threshold:.3f} px/frame, data-driven {REST_SPEED_QUANTILE:.0%} quantile\n')
    handle.write(f'- Active threshold: {active_threshold:.3f} px/frame, data-driven {ACTIVE_SPEED_QUANTILE:.0%} quantile\n\n')
    handle.write('## Headline summaries\n\n')
    handle.write(f'- Videos analyzed: {len(tracking_df)}\n')
    handle.write(f'- Median usable body-center coverage: {tracking_df["usable_center_pct"].median():.1f}%\n')
    handle.write(f'- Median total distance: {behavior_df["total_distance_px"].median():.1f} px\n')
    handle.write(f'- Median active frames: {behavior_df["active_frames_pct"].median():.1f}%\n')
    handle.write(f'- Videos flagged for manual review: {len(flags)}\n\n')
    handle.write('## Files\n\n')
    for name in [
        'tracking_qc_summary.csv',
        'preliminary_behavior_summary.csv',
        'bodypart_visibility_qc.csv',
        'manual_review_flags.csv',
        'tracking_coverage.png',
        'behavior_total_distance.png',
        'behavior_active_frames.png',
        'behavior_rest_frames.png',
        'tracking_bodypart_visibility_heatmap.png',
        'tracking_behavior_video_pages.pdf',
    ]:
        handle.write(f'- `{name}`\n')

print('Saved:', report)
print('Analysis complete. Output folder:', OUT)
